In [0]:
import pandas as pd                                                   # Import pandas for data cleaning
import numpy as np                                                    # Import Numpy for Maths functions
import matplotlib.pyplot as plt                                       # Import Matplot for Viz functions
import seaborn as sns                                                 # Import Seaborn for visualization
import matplotlib.pyplot as plt                                       # Import matplotlib library for visualization
import plotly.express as px                                           # Import plotly library for visualization
import plotly.graph_objects as go

from sklearn.ensemble import IsolationForest                          # EDA-Isolation Forest Analysis Functions

from pyspark.sql import functions as F
from pyspark.sql.functions import col, StringType, NumericType                  # TableFunctions
from pyspark.sql.functions import mean, min, max, stddev, count, sum as _sum    # MathsFunctions
from pyspark.sql.functions import to_date, year, month, datediff                # DateFunctions
from pyspark.sql.functions import abs                                           # OtherFunctions

from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

from sklearn.linear_model import LinearRegression                              # LinearRegression Analysis Functions
from sklearn.metrics import r2_score, mean_squared_error                       # LinearRegression Analysis Functions

from pyspark.ml.feature import StringIndexer, VectorAssembler, OneHotEncoder    # Classification Analysis Functions
from pyspark.ml import Pipeline                                                 # Classification Analysis Functions
from sklearn.linear_model import LogisticRegression                             # Classification Analysis Functions
from sklearn.metrics import mean_squared_error, r2_score                        # Classification Analysis Functions
from sklearn.impute import SimpleImputer                                        # Classification Analysis Functions
from sklearn.preprocessing import LabelEncoder                                  # Classification Analysis Functions
from sklearn.preprocessing import OneHotEncoder                                 # Classification Analysis Functions
from sklearn.preprocessing import StandardScaler                                # Classification Analysis Functions
from sklearn.model_selection import train_test_split                            # Classification Analysis Functions
from sklearn.metrics import accuracy_score, confusion_matrix                    # Classification Analysis Functions
from sklearn.metrics import classification_report                               # Classification Analysis Functions
from sklearn.metrics import roc_curve, auc                                      # Classification Analysis Functions

from sklearn.tree import DecisionTreeClassifier                                 # DecisionTree Analysis Functions

from sklearn.ensemble import RandomForestClassifier                             # RandomForest Analysis Functions

from sklearn.cluster import KMeans                                              # KMeans Cluster Analysis Functions


In [0]:
# Load dataset as Spark DataFrame

df = spark.table("indianhousing.bronze.indian_housing")
df_raw = df

# display(df)

In [0]:
df = spark.sql("""
SELECT 
    TRY_CAST(house_type AS STRING) AS house_type,
    TRY_CAST(regexp_replace(house_size,'[^0-9]','') AS INT) AS house_size,
    TRY_CAST(location AS STRING) AS location,
    TRY_CAST(city4 AS STRING) AS city,
    TRY_CAST(latitude AS DOUBLE) AS latitude,
    TRY_CAST(longitude AS DOUBLE) AS longitude,
    TRY_CAST(price AS INT) AS price,
    TRY_CAST(currency AS STRING) AS currency,
    COALESCE(
        TRY_CAST(numBathrooms AS INT),
        (SELECT percentile_approx(TRY_CAST(numBathrooms AS INT),0.5)
         FROM indianhousing.bronze.indian_housing),
        0
    ) AS num_Bathrooms,
    COALESCE(TRY_CAST(numBalconies AS INT),0) AS num_Balconies,
    TRY_CAST(COALESCE(isNegotiable,'NO') AS STRING) AS is_Negotiable,
    TRY_CAST(verificationDate AS STRING) AS verification_Date,
    TRY_CAST(
        CASE
            WHEN SecurityDeposit = 'No Deposit' THEN '0'
            ELSE SecurityDeposit
        END AS STRING
    ) AS Security_Deposit,
    TRY_CAST(Status AS STRING) AS Status
FROM indianhousing.bronze.indian_housing
""")

df.write.option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("indianhousing.silver.indian_housing")

# display(df)

In [0]:
# HANDLING NULLS
null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns
])
null_counts.write.mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable("indianhousing.silver.indian_housing_nulls")

# display(null_counts)

In [0]:
# Numeric columns to check
numeric_cols = [c for (c, t) in df.dtypes if t in ("int", "double", "float")]

# Function to add outlier flag for a column
def add_outlier_flag(df, col):
    q1, q3 = df.approxQuantile(col, [0.25, 0.75], 0.01)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    return df.withColumn(
        f"{col}_Outlier",
        F.when((F.col(col) < lower_bound) | (F.col(col) > upper_bound), 1).otherwise(0)
    )

# Apply outlier flagging for each numeric column
for col_name in numeric_cols:
    df = add_outlier_flag(df, col_name)

# Show sample with flags
df.select(numeric_cols + [f"{c}_Outlier" for c in numeric_cols]).show(1)

In [0]:
from pyspark.sql import functions as F
from sklearn.ensemble import IsolationForest
import pandas as pd

# Step 2: Select numeric columns for anomaly detection
# numeric_cols = ["house_size", "price", "num_Bathrooms", "num_Balconies"]

# Step 3: Convert numeric subset to Pandas
pdf = df.select(*numeric_cols).toPandas()

# Step 4: Fill missing values with median
for col in numeric_cols:
    pdf[col] = pdf[col].fillna(pdf[col].median())

# Step 5: Fit Isolation Forest
iso = IsolationForest(contamination=0.05, random_state=42)
pdf["is_anomaly"] = iso.fit_predict(pdf[numeric_cols])   # -1 = anomaly, 1 = normal
pdf["anomaly_score"] = iso.decision_function(pdf[numeric_cols])

# Step 6: Convert anomaly results back to Spark
df_anomaly = spark.createDataFrame(pdf)

# Step 7: Add anomaly columns back to full dataset
df_full = df.join(df_anomaly.select(*numeric_cols, "is_anomaly", "anomaly_score"),
                  on=numeric_cols, how="left")

# Step 8: Save full dataset with anomaly flags
spark.sql("DROP TABLE IF EXISTS indianhousing.silver.indian_housing_iforest")

df_full.write.option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("indianhousing.silver.indian_housing_EDA")

display(df_full)